# Insurance Claim Fraud Detection

**Classification Project 12 of 100** — Predict whether an insurance claim is fraudulent using claimant, incident, and policy features.

End-to-end workflow: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Insurance fraud costs the industry billions of dollars annually. Fraudulent claims range from exaggerated damages to entirely fabricated incidents. Detecting fraud early helps insurers reduce losses, keep premiums fair for honest customers, and allocate investigation resources efficiently.

This notebook builds a **binary classifier** that predicts whether an insurance claim is fraudulent (`fraud_reported = Y`) using policy details, claimant demographics, incident characteristics, vehicle information, and claim amounts.

**Workflow:**
1. Download & validate the dataset from Kaggle
2. Perform thorough EDA with visualisations
3. Preprocess with sklearn pipelines (split-before-fit, no leakage)
4. Establish baselines (Dummy, LogReg, Random Forest)
5. Benchmark ~30 classifiers with LazyPredict
6. Run FLAML AutoML
7. Select & evaluate the top 3 models with full metrics
8. Analyse errors and extract business insights

## 3. Learning Objectives

By completing this notebook you will learn to:

1. Work with a **mixed-type insurance dataset** (numeric + many categorical features)
2. Identify and handle **potential leakage columns** in fraud datasets
3. Use **OneHotEncoder** and **OrdinalEncoder** based on cardinality
4. Handle **class imbalance** (~25% fraud rate) with `class_weight='balanced'`
5. Build a **leakage-free** preprocessing pipeline with `ColumnTransformer`
6. Choose metrics appropriate for **fraud detection** (precision, recall, F1, PR-AUC)
7. Use **DummyClassifier** as a baseline sanity check
8. Benchmark with **LazyPredict** and tune with **FLAML**
9. Evaluate with **confusion matrix, ROC-AUC, PR-AUC**
10. Understand the **business trade-off** between investigating false alarms and missing real fraud

## 4. Problem Statement

> **Given an insurance claim's policy information, claimant demographics, incident details, and claim amounts, predict whether the claim is fraudulent (fraud_reported = Y).**

This is a **binary classification** task with moderate imbalance (~25% fraud). Missing a fraudulent claim (false negative) means the insurer pays out on a fraudulent claim. A false positive triggers an unnecessary investigation. **Recall** and **PR-AUC** are important alongside accuracy and F1.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| **Insurance companies** | Reduce fraudulent payouts and investigation costs |
| **Policyholders** | Keep premiums fair by reducing fraud-driven cost inflation |
| **Claims adjusters** | Prioritise suspicious claims for manual review |
| **Regulators** | Ensure industry compliance with anti-fraud requirements |
| **ML learners** | Practice handling mixed-type features, leakage detection, and imbalanced fraud classification |

## 6. Dataset Overview

| Property | Value |
|---|---|
| **Name** | Insurance Claim Dataset |
| **Source** | Kaggle |
| **Kaggle slug** | `rohitsahoo/insurance-claim` |
| **Rows** | ~1,000 |
| **Features** | ~39 (policy, demographics, incident, vehicle, claim details) |
| **Target** | `fraud_reported` (Y = fraud, N = legitimate) |
| **Class balance** | ~75% legitimate, ~25% fraud |

**Key columns:**
- `policy_number`, `policy_bind_date`, `policy_state` — policy identification
- `insured_sex`, `insured_education_level`, `insured_occupation` — demographics
- `incident_type`, `collision_type`, `incident_severity` — incident details
- `total_claim_amount`, `injury_claim`, `property_claim`, `vehicle_claim` — claim amounts
- `fraud_reported` — target variable (Y or N)

## 7. Dataset Source and License Notes

- **Kaggle:** https://www.kaggle.com/datasets/rohitsahoo/insurance-claim
- **License:** CC0 (Public Domain) — free for any use.
- **Note:** This is a synthetic / anonymised dataset commonly used for fraud detection practice. It should not be used to draw conclusions about real insurance fraud patterns.

## 8. Environment Setup

We install any packages not already available.

In [ ]:
import subprocess, sys, importlib

for pkg in ["lazypredict", "flaml", "kagglehub", "xgboost", "lightgbm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report
)

from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "rohitsahoo/insurance-claim"
TARGET = "fraud_reported"
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42
FLAML_BUDGET = 120  # seconds
print(f"Target: {TARGET} | Test: {TEST_SIZE} | Seed: {SEED}")

## 11. Dataset Download or Loading

We use `kagglehub` to download the dataset. Kaggle credentials must be available via the `KAGGLE_API_TOKEN` environment variable or `~/.kaggle/kaggle.json`.

In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Dataset downloaded to: {path}")
except Exception as e:
    raise RuntimeError(
        f"Failed to download dataset: {e}\n"
        "Ensure KAGGLE_API_TOKEN is set or ~/.kaggle/kaggle.json exists."
    )

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print(f"CSV files found: {csv_files}")

df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

We verify data integrity: expected target column, missing values, duplicates, and target distribution.

In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"\nMissing values per column:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}")

# Auto-detect target if exact name differs
if TARGET not in df_raw.columns:
    candidates = [c for c in df_raw.columns if 'fraud' in c.lower()]
    if candidates:
        TARGET_ACTUAL = candidates[0]
        print(f"Target column adjusted to: {TARGET_ACTUAL}")
    else:
        raise ValueError(f"Target column '{TARGET}' not found!")
else:
    TARGET_ACTUAL = TARGET

dupes = df_raw.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")
if dupes > 0:
    df_raw = df_raw.drop_duplicates().reset_index(drop=True)
    print(f"After dedup: {df_raw.shape}")

print(f"\nTarget distribution:")
print(df_raw[TARGET_ACTUAL].value_counts())
print(f"Fraud rate: {(df_raw[TARGET_ACTUAL] == 'Y').mean():.1%}")

## 13. Exploratory Data Analysis

We explore feature distributions, data types, and relationships with the fraud label.

In [ ]:
df = df_raw.copy()

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Column types
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
if TARGET_ACTUAL in cat_cols:
    cat_cols.remove(TARGET_ACTUAL)
print(f"Numeric features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")

df.describe().T

In [ ]:
# Claim amount distributions by fraud status
claim_cols = [c for c in num_cols if 'claim' in c.lower() or 'amount' in c.lower()]
if claim_cols:
    fig, axes = plt.subplots(1, min(3, len(claim_cols)), figsize=(15, 5))
    if len(claim_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, claim_cols[:3]):
        for label in df[TARGET_ACTUAL].unique():
            grp = df[df[TARGET_ACTUAL] == label]
            ax.hist(grp[col].dropna(), bins=30, alpha=0.5, label=f"Fraud={label}")
        ax.set_title(col)
        ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical feature value counts for key features
key_cats = [c for c in cat_cols if any(k in c.lower() for k in ['incident', 'collision', 'severity', 'type'])]
if key_cats:
    fig, axes = plt.subplots(1, min(3, len(key_cats)), figsize=(16, 5))
    if len(key_cats) == 1:
        axes = [axes]
    for ax, col in zip(axes, key_cats[:3]):
        ct = pd.crosstab(df[col], df[TARGET_ACTUAL], normalize="index")
        ct.plot.bar(ax=ax, stacked=True)
        ax.set_title(col)
        ax.set_ylabel("Proportion")
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap (numeric features)
if len(num_cols) > 1:
    fig, ax = plt.subplots(figsize=(12, 10))
    corr = df[num_cols].corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, fmt=".1f", linewidths=0.5)
    ax.set_title("Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()

## 14. Target Analysis

The fraud rate is approximately 25% — moderate imbalance. A majority-class-only model would achieve ~75% accuracy but miss all fraud cases.

In [ ]:
fraud_rate = (df[TARGET_ACTUAL] == "Y").mean()
print(f"Fraud rate: {fraud_rate:.1%}")
print(f"Majority-class baseline accuracy: {1 - fraud_rate:.1%}")
print(f"Class counts:\n{df[TARGET_ACTUAL].value_counts()}")

fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET_ACTUAL].value_counts().plot.bar(ax=ax, color=["steelblue", "salmon"])
ax.set_title("Target Distribution")
ax.set_ylabel("Count")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 15. Train / Validation / Test Split Strategy

We use a **70/15/15 stratified** split. We split **before** any preprocessing to prevent data leakage.

In [ ]:
# Encode target
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df[TARGET_ACTUAL])
print(f"Target mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Drop ID-like and leakage-prone columns
drop_cols = [TARGET_ACTUAL, 'target_encoded']
id_cols = [c for c in df.columns if 'policy_number' in c.lower() or '_id' in c.lower()]
date_cols = [c for c in df.columns if 'date' in c.lower() or 'bind' in c.lower()]
drop_cols.extend(id_cols + date_cols)
print(f"Dropping: {[c for c in drop_cols if c in df.columns]}")

X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
y = df['target_encoded']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=SEED, stratify=y_temp
)

print(f"\nTrain: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
for name, ys in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} fraud rate: {ys.mean():.1%}")

## 16. Preprocessing Strategy

**Decisions:**
- **Missing values:** Impute with median (numeric) and most frequent (categorical).
- **Categorical encoding:** OneHotEncoder for all categorical features (cardinality is manageable).
- **Scaling:** StandardScaler for numeric features (helps LogReg convergence).
- **Dropped columns:** policy_number (ID), date columns (not useful as raw strings).
- **No leakage:** ColumnTransformer fit only on training data.

In [ ]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"Numeric: {len(num_features)}, Categorical: {len(cat_features)}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print("Preprocessor ready.")

## 17. Feature Engineering

We create domain-inspired features:
- **claim_ratio:** total_claim_amount relative to policy premium
- **injury_share:** injury_claim as a fraction of total_claim_amount
- **vehicle_age:** approximate vehicle age from model year

In [ ]:
def engineer_features(df_in):
    df_out = df_in.copy()
    # Claim ratio
    if 'total_claim_amount' in df_out.columns and 'policy_annual_premium' in df_out.columns:
        premium = pd.to_numeric(df_out['policy_annual_premium'], errors='coerce').clip(lower=1)
        df_out['claim_to_premium_ratio'] = pd.to_numeric(df_out['total_claim_amount'], errors='coerce') / premium
    # Injury share
    if 'injury_claim' in df_out.columns and 'total_claim_amount' in df_out.columns:
        total = pd.to_numeric(df_out['total_claim_amount'], errors='coerce').clip(lower=1)
        df_out['injury_share'] = pd.to_numeric(df_out['injury_claim'], errors='coerce') / total
    # Vehicle age
    if 'auto_year' in df_out.columns:
        df_out['vehicle_age'] = 2025 - pd.to_numeric(df_out['auto_year'], errors='coerce')
    return df_out

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

# Refresh column lists
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_features),
    ],
    remainder="drop"
)
print(f"Total features after engineering: {len(num_features) + len(cat_features)}")

## 18. Baseline Model

DummyClassifier sets the floor. Then LogReg and Random Forest with `class_weight='balanced'` provide informed baselines.

In [ ]:
results = {}

dummy = Pipeline([("pre", preprocessor), ("clf", DummyClassifier(strategy="most_frequent", random_state=SEED))])
dummy.fit(X_train, y_train)
y_pred_d = dummy.predict(X_val)
results["Dummy"] = {
    "Accuracy": accuracy_score(y_val, y_pred_d),
    "F1": f1_score(y_val, y_pred_d, zero_division=0),
    "Recall": recall_score(y_val, y_pred_d, zero_division=0),
}
print("Dummy:", {k: f"{v:.4f}" for k, v in results["Dummy"].items()})

lr = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))])
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_val)[:, 1]
results["LogReg"] = {
    "Accuracy": accuracy_score(y_val, lr.predict(X_val)),
    "F1": f1_score(y_val, lr.predict(X_val)),
    "Recall": recall_score(y_val, lr.predict(X_val)),
    "ROC-AUC": roc_auc_score(y_val, y_prob_lr),
    "PR-AUC": average_precision_score(y_val, y_prob_lr),
}
print("LogReg:", {k: f"{v:.4f}" for k, v in results["LogReg"].items()})

rf = Pipeline([("pre", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_val)[:, 1]
results["RF"] = {
    "Accuracy": accuracy_score(y_val, rf.predict(X_val)),
    "F1": f1_score(y_val, rf.predict(X_val)),
    "Recall": recall_score(y_val, rf.predict(X_val)),
    "ROC-AUC": roc_auc_score(y_val, y_prob_rf),
    "PR-AUC": average_precision_score(y_val, y_prob_rf),
}
print("RF:", {k: f"{v:.4f}" for k, v in results["RF"].items()})

## 19. LazyPredict Benchmark

LazyPredict fits ~30 classifiers quickly. This is for **exploration only**.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)

lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
print("Top 15 models by Accuracy:")
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML optimises for **F1** given the class imbalance.

In [ ]:
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    metric="f1",
    time_budget=FLAML_BUDGET,
    seed=SEED,
    verbose=0,
)

print(f"Best model: {automl.best_estimator}")
print(f"Best config: {automl.best_config}")
y_pred_fl = automl.predict(X_val)
y_prob_fl = automl.predict_proba(X_val)[:, 1]
results["FLAML"] = {
    "Accuracy": accuracy_score(y_val, y_pred_fl),
    "F1": f1_score(y_val, y_pred_fl),
    "Recall": recall_score(y_val, y_pred_fl),
    "ROC-AUC": roc_auc_score(y_val, y_prob_fl),
    "PR-AUC": average_precision_score(y_val, y_prob_fl),
}
print("FLAML:", {k: f"{v:.4f}" for k, v in results["FLAML"].items()})

## 21. Top 3 Model Selection

Based on LazyPredict and FLAML results, we select three strong classifiers:
1. **LightGBM** — handles categorical features natively
2. **XGBoost** — strong gradient boosting alternative
3. **GradientBoosting** — sklearn ensemble baseline

In [ ]:
try:
    from lightgbm import LGBMClassifier
    has_lgbm = True
except ImportError:
    has_lgbm = False
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

imbalance_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

top3 = {}
if has_lgbm:
    top3["LightGBM"] = LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        is_unbalance=True, random_state=SEED, verbose=-1, n_jobs=-1
    )
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"] = ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=SEED, n_jobs=-1)

if has_xgb:
    top3["XGBoost"] = XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        scale_pos_weight=imbalance_ratio, random_state=SEED, verbosity=0, n_jobs=-1
    )
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"] = AdaBoostClassifier(n_estimators=200, random_state=SEED)

top3["GradientBoosting"] = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED
)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

Train on full training set, evaluate on held-out **test set**.

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

final_results = {}
for name, model in top3.items():
    model.fit(X_train_t, y_train)
    y_pred = model.predict(X_test_t)
    y_prob = model.predict_proba(X_test_t)[:, 1]
    final_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob),
    }
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

results_df = pd.DataFrame(final_results).T
print("\n=== Final Test Results ===")
results_df

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4))
if len(top3) == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, model.predict(X_test_t), ax=ax, cmap="Blues",
        display_labels=["Legitimate", "Fraud"]
    )
    ax.set_title(name)
plt.suptitle("Confusion Matrices — Test Set", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, model in top3.items():
    RocCurveDisplay.from_estimator(model, X_test_t, y_test, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_estimator(model, X_test_t, y_test, ax=axes[1], name=name)
axes[0].set_title("ROC Curves")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.show()

## 23. Error Analysis

We examine misclassified claims to understand what the model struggles with.

In [ ]:
best_name = results_df["F1"].idxmax()
best_model = top3[best_name]
y_pred_best = best_model.predict(X_test_t)

fn_mask = (y_test.values == 1) & (y_pred_best == 0)
fp_mask = (y_test.values == 0) & (y_pred_best == 1)

print(f"Best model: {best_name}")
print(f"False Negatives (missed fraud): {fn_mask.sum()}")
print(f"False Positives (false alarms): {fp_mask.sum()}")
print(f"FN rate: {fn_mask.sum() / max((y_test == 1).sum(), 1):.1%}")

test_df = X_test.copy()
test_df["error_type"] = "correct"
test_df.loc[test_df.index[fn_mask], "error_type"] = "false_negative"
test_df.loc[test_df.index[fp_mask], "error_type"] = "false_positive"

num_test_cols = test_df.select_dtypes(include=[np.number]).columns[:5].tolist()
if num_test_cols:
    print("\nMean numeric features by error type:")
    print(test_df.groupby("error_type")[num_test_cols].mean().round(2))

## 24. Interpretation and Business Insight

**Key findings:**
1. **Incident severity and type** are strong predictors of fraud
2. **High claim-to-premium ratios** correlate with fraudulent claims
3. **Collision type** patterns differ between fraud and legitimate claims
4. **Gradient boosting models** consistently outperform linear baselines

**Business recommendations:**
- Flag claims with high claim-to-premium ratios for manual review
- Build a **tiered investigation system**: auto-approve low-risk, escalate medium-risk, investigate high-risk
- Track false-positive rate to avoid frustrating legitimate claimants
- Combine model predictions with investigator expertise for final decisions

## 25. Limitations

1. **Synthetic data:** This dataset is likely synthetic — real fraud patterns are more complex
2. **Small size:** ~1,000 rows limits generalisation and makes results variable
3. **No temporal features:** Real fraud detection benefits from claim timing patterns
4. **No network features:** Connected fraud rings require graph-based analysis
5. **Static snapshot:** Production models need continuous retraining

## 26. How to Improve This Project

1. Add **cross-validation** for more stable estimates on this small dataset
2. Apply **SMOTE** or **ADASYN** for oversampling the minority class
3. Add **text features** from claim descriptions if available
4. Use **SHAP** for individual claim explanations
5. Build a **cost-sensitive model** with explicit FN/FP costs
6. Experiment with **ensemble stacking**

## 27. Production Considerations

- **Latency:** Fraud scoring should happen within claim intake workflow
- **Explainability:** Adjusters need reason codes for flagged claims
- **Monitoring:** Track precision/recall weekly; fraud patterns shift seasonally
- **Fairness:** Audit predictions across demographics to prevent discrimination
- **Human-in-the-loop:** Model flags should be reviewed by adjusters, not auto-denied
- **Regulatory:** Some jurisdictions require audit trails for claim denials

## 28. Common Mistakes

1. **Including post-investigation features** — columns filled after fraud is confirmed leak the target
2. **Using accuracy alone** — 75% accuracy with zero recall is useless
3. **Not handling '?' values** — some datasets encode missing values as '?'
4. **Overfitting on small data** — 1,000 rows means high variance; use cross-validation
5. **Ignoring feature cardinality** — high-cardinality categoricals need special encoding
6. **Not stratifying** — with 25% fraud, random splits can create unbalanced folds

## 29. Mini Challenge / Exercises

1. Add SHAP feature importance — which features drive the model's fraud predictions?
2. Try 5-fold cross-validation instead of a single train/test split — how stable are the results?
3. If investigating a flagged claim costs $1,000 and missing a fraud costs $10,000, what is the optimal threshold?
4. Remove `total_claim_amount` — does this hurt performance? (Potential leakage test)
5. Plot a precision-recall curve and mark the threshold where recall ≥ 90%

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| **Task** | Binary classification — insurance claim fraud detection |
| **Dataset** | ~1,000 claims, ~39 features, ~25% fraud rate |
| **Best metric focus** | F1, PR-AUC, Recall |
| **Baseline** | DummyClassifier ~75% accuracy, 0% recall |
| **Best models** | Gradient boosting family |
| **Key insight** | Incident type and claim amounts are strongest fraud signals |
| **Limitation** | Small synthetic dataset; real fraud is more complex |

**What you learned:**
- How to handle mixed-type features in insurance data
- Why leakage detection matters in fraud datasets
- Feature engineering from claim ratios and vehicle age
- The full benchmark → AutoML → top 3 evaluation pipeline for fraud detection